# 00 - Cryptic IP Pipeline: Colab Quickstart

This notebook clones the `cryptic-ip-pipeline` repository, installs the
`crypticip` Python package, and verifies the install with the test
suite and a synthetic smoke workflow.

**What this notebook does NOT do:**
- It does not install the external scientific binaries
  (`fpocket`, `freesasa`, `pdb2pqr`, `apbs`, `pymol`). Those are NOT
  preinstalled on vanilla Colab and require root/conda or Docker to
  set up. See notebook `01_validation_colab.ipynb` for installation
  options. Without them the screening pipeline runs in **fallback
  mode** and produces zero pockets.
- It does not download any AlphaFold proteomes. See
  `02_yeast_screening_colab.ipynb` for that.

**Storage:** < 100 MB. **Time:** ~3-5 min on a free Colab CPU runtime.


## 1. (Optional) Mount Google Drive

Useful if you want to persist outputs across sessions.

In [ ]:
MOUNT_DRIVE = False  # set True to mount /content/drive
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')


## 2. Clone the repository

In [ ]:
import os, subprocess, pathlib
REPO_URL = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
WORK = pathlib.Path('/content/cryptic-ip-pipeline')
if not WORK.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK)], check=True)
os.chdir(WORK)
print('cwd:', os.getcwd())
print(sorted(p.name for p in WORK.iterdir())[:20])


## 3. Install the Python package

Installs `crypticip` in editable mode along with required Python deps.

In [ ]:
!pip install -q -e .
!pip install -q nbformat  # for notebook-level checks below


## 4. Verify the install

In [ ]:
!crypticip --version

In [ ]:
!crypticip --help

## 5. `check-env` (expected: most external tools missing on Colab)

Colab images do not ship with fpocket / freesasa / pdb2pqr / apbs / pymol.
All five will be reported missing. This is fine for this quickstart;
the next notebooks show how to install them where possible.

In [ ]:
!crypticip check-env

## 6. Run the test suite

In [ ]:
!python -m pytest -q

## 7. Synthetic mini-proteome smoke workflow

Builds a 3-file 'fake yeast' proteome (using the test fixture
`tests/fixtures/tiny.pdb`) and runs the full CLI sequence end-to-end
in **fallback mode**. Expected: `n_pockets = 0`, `n_rows = 0` — the
fpocket-free path. The point is to confirm the plumbing, not the
biology.

In [ ]:
import shutil, pathlib, yaml
SMOKE = pathlib.Path('/content/crypticip_smoke')
PROT = SMOKE / 'data/proteomes/yeast'
PROT.mkdir(parents=True, exist_ok=True)
for acc in ('P00001', 'P00002', 'P00003'):
    shutil.copy('tests/fixtures/tiny.pdb', PROT / f'AF-{acc}-F1-model_v4.pdb')
cfg = {
    'paths': {
        'data_dir': str(SMOKE/'data'),
        'proteomes_dir': str(SMOKE/'data/proteomes'),
        'cache_dir': str(SMOKE/'data/cache'),
        'results_dir': str(SMOKE/'results'),
        'reports_dir': str(SMOKE/'results/reports'),
        'screening_dir': str(SMOKE/'results/screening'),
        'experimental_dir': str(SMOKE/'results/experimental'),
    },
    'organism': 'yeast',
    'proteome': {
        'uniprot_id': 'UP000002311',
        'scientific_name': 'Saccharomyces cerevisiae',
        'alphafold_tar_url': 'https://example.invalid/yeast.tar',
        'expected_file_count_min': 1, 'expected_file_count_max': 100,
        'taxonomy_id': 559292,
    },
    'screening': {'mode': 'fast', 'workers': 1},
}
cfg_path = SMOKE / 'smoke_config.yaml'
cfg_path.write_text(yaml.safe_dump(cfg))
print('config:', cfg_path)


In [ ]:
!crypticip index-proteome --organism yeast --config /content/crypticip_smoke/smoke_config.yaml

In [ ]:
!crypticip screen --organism yeast --mode fast --workers 1 --limit 3 --config /content/crypticip_smoke/smoke_config.yaml

In [ ]:
!crypticip report --organism yeast --config /content/crypticip_smoke/smoke_config.yaml

In [ ]:
!crypticip pymol --organism yeast --top 5 --config /content/crypticip_smoke/smoke_config.yaml

In [ ]:
!crypticip experimental-plan --organism yeast --top 5 --config /content/crypticip_smoke/smoke_config.yaml

## 8. What you just saw

- Tests pass.
- The CLI iterates over the synthetic proteome, but because **fpocket
  is not installed**, `screening_results.csv` is empty (no pockets
  detected). PyMOL session generation and experimental-plan files
  succeed but contain no candidates.

**Next:** open `01_validation_colab.ipynb` for installing the
external binaries and running validation on the curated controls.